# 5. Greeks by adjoint algorithmic differentiation (AAD)

**Level:** L2. **Tags:** `aad`, `greeks`, `black-scholes`.
**Library:** `qp_core.aad` (`python/core`).

**Plain idea.** A pricer is a long chain of simple operations. Record the
chain once on a "tape" while pricing, then walk it backwards with the chain
rule: one backward pass gives the derivative of the price with respect to
every input at once, for a small constant multiple of the cost of one price.
Bumping each input and repricing costs one extra price per input instead.

**Formal statement** (Griewank and Walther, *Evaluating Derivatives*, 2nd
ed., 2008, ch. 3; Giles and Glasserman, "Smoking adjoints", *Risk*, 2006):
for $y = f(x_1, \dots, x_n)$ built from elementary operations
$v_j = \phi_j(v_{i \prec j})$, set the adjoint $\bar y = 1$ and run
$\bar v_i \mathrel{+}= \bar v_j\, \partial\phi_j/\partial v_i$ over the tape in
reverse order; then $\bar x_i = \partial y / \partial x_i$.

In [ ]:
import _paths  # noqa: F401  (puts the library on sys.path)

In [ ]:
import math

from qp_core.aad.black_scholes import bs_greeks_aad, bs_price
from scipy.stats import norm

## Task 1: all first-order Greeks from one backward pass

Call `bs_greeks_aad` for a call with $S = 100$, $K = 105$, $T = 0.5$,
$r = 3\%$, $q = 1\%$, $\sigma = 25\%$. Store the result as `g`.

In [ ]:
S, K, T, r, q, sigma = 100.0, 105.0, 0.5, 0.03, 0.01, 0.25
g = bs_greeks_aad(S, K, T, r, q, sigma)
print(f"price {g.price:.6f} delta {g.delta:.6f} vega {g.vega:.6f} rho {g.rho:.6f}")

**Independent check.** The textbook closed forms (Hull, 11th ed., the
Greeks chapter): $\Delta = e^{-qT}N(d_1)$,
$\mathcal V = S e^{-qT}\varphi(d_1)\sqrt T$, $\rho = K T e^{-rT} N(d_2)$.
They share no code with the tape.

In [ ]:
d1 = (math.log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
d2 = d1 - sigma * math.sqrt(T)
price = S * math.exp(-q * T) * norm.cdf(d1) - K * math.exp(-r * T) * norm.cdf(d2)
assert abs(g.price - price) < 1e-12
assert abs(g.delta - math.exp(-q * T) * norm.cdf(d1)) < 1e-12
assert abs(g.vega - S * math.exp(-q * T) * norm.pdf(d1) * math.sqrt(T)) < 1e-10
assert abs(g.rho - K * T * math.exp(-r * T) * norm.cdf(d2)) < 1e-10

## Task 2: the same Greeks by bump and reprice

Central differences with a bump of `1e-5` in $S$ and in $\sigma$, using the
plain-float `bs_price(S, K, T, r, q, sigma)`. Store `delta_fd` and
`vega_fd`. That is two extra prices per Greek; AAD needed one backward pass
for all seven numbers.

In [ ]:
h = 1e-5
delta_fd = (bs_price(S + h, K, T, r, q, sigma) - bs_price(S - h, K, T, r, q, sigma)) / (2 * h)
vega_fd = (bs_price(S, K, T, r, q, sigma + h) - bs_price(S, K, T, r, q, sigma - h)) / (2 * h)
print(f"delta fd {delta_fd:.8f}  vega fd {vega_fd:.6f}")

In [ ]:
assert abs(delta_fd - g.delta) < 1e-6
assert abs(vega_fd - g.vega) < 1e-5

**Takeaways.** AAD is exact to rounding, not to a bump size, and its cost
does not grow with the number of inputs. The Rust and C++ versions and their
measured cost ratios are in `rust/aad/README.md` and `cpp/aad/README.md`.